# Session 9 lab: group, join and reshape

**Question of the day:** How do we answer a question that needs several groups and several tables, and trust the total?

In every step: **predict** first, run the code, **read** the output, then **change** one thing. You keep this notebook. It ends with a join log (the row count before and after every merge) and one written finding.

Data: Lakeside Mart (fictional, generated). The orders file is a 5 percent sample of till orders, so revenue describes the sample.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
import pandas as pd

orders = pd.read_csv('orders.csv')
lines = pd.read_csv('order_lines.csv')
branches = pd.read_csv('branches.csv')
customers = pd.read_csv('customers.csv')
products = pd.read_csv('products.csv')
print(len(orders), len(lines))

# The join log: one row for every merge. Run this cell once. You do not need to read it.
join_log = []
def log(step, rows_before, rows_after, note):
    join_log.append(dict(step=step, rows_before=rows_before, rows_after=rows_after, note=note))
def safe(f):
    try:
        return f()
    except Exception:
        return -1   # lets a check say NOT YET instead of stopping

---
## Step 1: groupby and named aggregation

groupby splits the records into groups. An aggregate function reduces each group to one number.

**PREDICT first.** Add up order_value_ugx for each branch. Will the eight branch totals add up to the total of the whole orders file? Which branch will lead?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
rev_branch = None   # group orders by branch_id, then sum order_value_ugx
summary = None      # named aggregation: orders (count of order_id), revenue (sum), average (mean)

# Automatic check
check('B01 revenue', safe(lambda: rev_branch['B01']), 283331305)
check('eight branches add up to the total', safe(lambda: rev_branch.sum()), 1435614185)
check('B01 orders in summary', safe(lambda: summary.loc['B01', 'orders']), 5885)

**READ the output.** Which branch has the largest revenue? Does it also have the largest average order? Look at the summary table.

> My answer: 

**CHANGE one thing.** Add a fourth named aggregation, biggest=('order_value_ugx', 'max'). What is the largest order of B01?

> What I saw: 

---
## Step 2: count and size

size counts records. count counts only filled values. Walk-in orders have a missing customer_id.

**PREDICT first.** For branch B01, will the number of records and the number of filled customer_id values be equal? If not, which is smaller?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
sizes = None       # groupby branch_id, agg: records=('customer_id', 'size'), filled=('customer_id', 'count')
b01_filled = None  # the filled value for B01

# Automatic check
check('B01 filled customer_id', safe(lambda: b01_filled), 4071)

**READ the output.** Why is filled smaller than records? Write the reason: the blank means a walk-in shopper.

> My answer: 

---
## Step 3: A safe lookup: left merge, many-to-one, row counts

merge adds the branch details to every order. Many-to-one with how=left should keep every order.

**PREDICT first.** orders has 30,229 rows. How many rows will the merged table have?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
named = None   # merge branches onto orders: on='branch_id', how='left', validate='m:1'
# then add a line to the join log: log('orders + branches (left, m:1)', len(orders), len(named), 'every order keeps one row')

# Automatic check
check('rows after the left merge', safe(lambda: len(named)), 30229)

**CHANGE one thing.** Delete validate='m:1' and run again. Does the row count change? Then put it back. validate costs nothing and protects you.

> What I saw: 

---
## Step 4: Lost rows: inner join and left join

An inner join keeps only keys found in both tables. Orders with no customer_id have no match.

**PREDICT first.** Merge orders to customers on customer_id with an inner join. How many orders will be lost? Why?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
inner = None   # inner join of orders and customers on customer_id, validate='m:1'
left = None    # the same with how='left'
n_lost = None  # rows of orders that the inner join lost
# then two lines for the join log, as in step 3

# Automatic check
check('orders lost by the inner join', safe(lambda: n_lost), 9171)
check('left join keeps every order', safe(lambda: len(left)), 30229)

**READ the output.** The lost orders have no customer_id. What share of the orders is that, and is losing them acceptable for a report on total sales?

> My answer: 

**CHANGE one thing.** Repeat the left merge with indicator=True and run left['_merge'].value_counts(). How many orders show left_only? Which number from this step does it equal?

> What I saw: 

---
## Step 5: Fan-out: the merge that makes revenue too big

A one-to-many merge repeats the order once for every line. Adding order_value_ugx then counts it once per line.

**PREDICT first.** Merge orders to order_lines on order_id with no validate. The lines average 3.2 per order. Roughly how many times too big will the revenue be?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
fan = None         # merge orders and lines on order_id, with no validate
inflated = None    # sum of order_value_ugx in fan
true_total = None  # sum of order_value_ugx in orders
ratio = None       # inflated divided by true_total
# then a join log line

# Automatic check
check('rows after the fan-out merge', safe(lambda: len(fan)), 98066)
check('inflated revenue', safe(lambda: inflated), 7606090055)
check('ratio', safe(lambda: ratio), 5.2981, 0.01)

**READ the output.** Compare the ratio (about 5.3) with the average number of lines per order (about 3.2). Why is the ratio larger? Hint: which orders have the most lines?

> My answer: 

---
## Step 6: validate refuses the bad merge

validate='m:1' says the right table must have each key once. order_lines repeats order_id, so pandas raises a MergeError.

**PREDICT first.** Will orders.merge(lines, on='order_id', validate='m:1') run, or raise an error?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
message = None   # run the merge with validate='m:1' inside try ... except pd.errors.MergeError as err, and keep str(err)

# Automatic check
check('validate refused the merge', safe(lambda: int('not a many-to-one merge' in message)), 1)

**CHANGE one thing.** Change validate to '1:m' (one-to-many). Does it run now? Why? Compare len() of the result with the fan-out table from step 5.

> What I saw: 

---
## Step 7: One table with category and month, using two many-to-one merges

Start from the many side (lines) and look up the one side twice. The row count must stay the same.

**PREDICT first.** order_lines has 98,066 rows. How many rows will full have after two many-to-one merges? Will the revenue change?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
full = None   # lines + category from products (on product_id), then branch_id and order_date from orders (on order_id), both m:1
# then: order_date with pd.to_datetime, a month column with .dt.strftime('%Y-%m'), and a join log line

# Automatic check
check('rows in full', safe(lambda: len(full)), 98066)
check('revenue unchanged', safe(lambda: full['line_revenue_ugx'].sum()), 1435614185)

**READ the output.** Why does the revenue of full equal the revenue of the orders file?

> My answer: 

---
## Step 8: Dates: the weekday of every order

Convert text to a datetime first. Then .dt.day_name() gives the weekday.

**PREDICT first.** Which weekday will have the highest revenue? Write a guess.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
by_day = None    # make a weekday variable, group by it, sum order_value_ugx
top_day = None   # the weekday with the largest total (idxmax)

# Automatic check
check('revenue of the top weekday', safe(lambda: by_day['Thursday']), 223743120)
check('top weekday', safe(lambda: int(top_day == 'Thursday')), 1)

**READ the output.** How many times larger is the highest weekday than the lowest? Is that a pattern, or could it be chance? Write what you would need to know.

> My answer: 

---
## Step 9: pivot_table: region by year

index is the variable down the side, columns is the variable across the top, margins=True adds the totals (All).

**PREDICT first.** The bottom right cell of the pivot should equal one number you already know. Which one?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
pivot = None   # merge region onto orders (m:1), add a year variable, then pivot_table with margins=True

# Automatic check
check('Central 2024', safe(lambda: pivot.loc['Central', 2024]), 334294220)
check('grand total', safe(lambda: pivot.loc['All', 'All']), 1435614185)

**CHANGE one thing.** Put payment_method across the top instead of year. What is the grand total now?

> What I saw: 

---
## Step 10: Keep: the join log and one finding

Your join log shows which merges kept, lost or multiplied rows. Write one finding with a number and its source table.

In [ ]:
log_table = None   # pd.DataFrame(join_log), then print it
finding = None      # one sentence with a number and its source table

# Automatic check
check('join log has five rows', safe(lambda: len(log_table)), 5)
check('finding written', safe(lambda: int(len(finding) > 40)), 1)

**Keep this notebook.** Save a copy to your Drive (File, Save a copy in Drive). Bring one merge from your own work to Session 10.

In [ ]:
show_checks()